# Tutorial 1, Wednesday 7 October

This is the complete Lecture 3 notebook with space for today's exercises at the bottom. **Run All first** (it defines `p`, `ages`, `solve_household_grid` and `simulate_profile`), then work in the cells under *Tutorial 1* at the end. To change one parameter: `q = {**p, "n_a": 30}`.

# Lecture 3: a household that dies

**ECU44291 Quantitative Macroeconomics, Week 3.**

A household lives $J$ periods, works for the first $J_R$, earns $w e_j$ (an age-efficiency profile), pays a contribution rate $\tau$ on earnings, gets a pension $b$ when retired, saves at gross return $R$, cannot borrow, and starts and ends with nothing:

$$V_j(a) = \max_{0 \le a' \le R a + y_j} \; u(Ra + y_j - a') + \beta V_{j+1}(a'), \qquad V_{J+1} \equiv 0, \qquad u(c) = \frac{c^{1-\sigma}}{1-\sigma}.$$

The terminal condition makes this a **finite algorithm**: solve age $J$, then $J-1$, and so on. No convergence loop. This is the basic form of M1 Part A.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = dict(J=60, J_R=45, beta=0.96, R=1.04, sigma=2.0, w=1.0, tau=0.0, repl=0.4, a_max=25.0, n_a=300)
ages = 20 + np.arange(1, p["J"] + 1)

## Task 1: the income profile

Write `efficiency_profile(J, J_R)` returning a hump-shaped $e_j$ (for example $\exp(0.06 j - 0.0012 j^2)$, normalised to mean one over working ages, zero after retirement) and `income_profile(p)` returning $y_j = (1-\tau) w e_j$ while working and a pension equal to `repl` times average gross earnings when retired. Plot it.

In [ ]:
# SOLUTION
def efficiency_profile(J, J_R):
    j = np.arange(1, J + 1)
    e = np.exp(0.06 * j - 0.0012 * j**2)
    e = e / e[:J_R].mean()
    e[J_R:] = 0.0
    return e

def income_profile(p):
    e = efficiency_profile(p["J"], p["J_R"])
    y = (1 - p["tau"]) * p["w"] * e
    y[p["J_R"]:] = p["repl"] * p["w"] * e[:p["J_R"]].mean()
    return y

y = income_profile(p)
plt.plot(ages, y); plt.xlabel("age"); plt.ylabel("income"); plt.show()

## Task 2: backward induction on a grid

Write `solve_household_grid(p)` returning `a_grid`, `V` (shape `(J+1, n_a)`, last row zero) and `policy` (shape `(J, n_a)`, the index of $a'$). Inside the age loop, build the matrix of consumption for every (today's assets, choice) pair, set infeasible entries to $-\infty$, add $\beta V_{j+1}$, and take the max and argmax along the choice axis.

Say out loud, before you run it: which direction does the loop run, and why?

In [ ]:
# SOLUTION
def utility(c, sigma):
    return np.log(c) if sigma == 1.0 else c ** (1 - sigma) / (1 - sigma)

def solve_household_grid(p):
    J, R, beta, sigma = p["J"], p["R"], p["beta"], p["sigma"]
    a = np.linspace(0.0, p["a_max"], p["n_a"])
    y = income_profile(p)
    V = np.zeros((J + 1, len(a)))                  # V[J] = 0 is the terminal condition
    policy = np.zeros((J, len(a)), dtype=int)
    for j in range(J - 1, -1, -1):                 # last age first
        c = R * a[:, None] + y[j] - a[None, :]     # rows: a today; columns: a' choices
        u = np.full(c.shape, -np.inf)
        u[c > 0] = utility(c[c > 0], sigma)
        value = u + beta * V[j + 1][None, :]
        policy[j] = value.argmax(axis=1)
        V[j] = value.max(axis=1)
    return dict(a_grid=a, V=V, policy=policy, y=y)

sol = solve_household_grid(p)
print("last-age policy saves nothing:", np.all(sol["policy"][-1] == 0))

**Workflow note.** *Print statements are a debugger you have to clean up afterwards.* Step through your solver in the VS Code debugger instead:

1. Click in the gutter to the left of the line in your age loop that assigns `V[j]`. A red dot appears.
2. Run the cell that calls `solve_household_grid` with **Debug Cell**: the small arrow next to the cell's Run button, or *Jupyter: Debug Cell* from the Command Palette (Ctrl+Shift+P, Cmd+Shift+P on a Mac). Shift+Enter and the Run button ignore breakpoints: the cell just runs to the end.
3. Execution stops on that line. Look at `j` in the Variables panel: which age is solved first?
4. Press F10 to run the line, then select the text `V[j]` and hover over it (or type `V[j]` in the Debug Console). That row of `n_a` numbers is the value function at one age: a vector, one value per point on the asset grid.
5. Press F5 to go round the loop again, and watch the order in which the rows of `V` fill in. Press Stop on the debug toolbar when you have seen enough.

## Task 3: look at what you solved

Plot $V_j(a)$ for ages 25, 45, 65, 75 on one figure, and the policies $a'(a)$ for the same ages with the 45-degree line. At which ages is $V$ higher, and why is that not a statement about who is better off? (Hint: with $\sigma = 2$, what is the sign of $u(c)$?) Where is $V$ steepest, and what does the slope measure? Why is the policy at 75 below the 45-degree line everywhere?

In [ ]:
# SOLUTION
a, V, pol = sol["a_grid"], sol["V"], sol["policy"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j in (4, 24, 44, 54):
    axes[0].plot(a, V[j], label=f"age {20 + j + 1}")
    axes[1].plot(a, a[pol[j]], label=f"age {20 + j + 1}")
axes[1].plot(a, a, "k--", lw=0.8)
axes[0].set_xlabel("a"); axes[0].set_ylabel("V"); axes[0].legend()
axes[1].set_xlabel("a"); axes[1].set_ylabel("a'"); plt.show()

## Task 4: simulate a life

Write `simulate_profile(sol, p)` that starts at zero assets and follows the policy, returning assets (length $J+1$) and consumption (length $J$). Plot consumption against income, and assets by age. Three things to explain: why consumption is flat in the middle of life, why it tracks income early, and why assets peak at retirement.

In [ ]:
# SOLUTION
def simulate_profile(sol, p):
    a, pol, y, R = sol["a_grid"], sol["policy"], sol["y"], p["R"]
    idx, assets, cons = 0, [0.0], []
    for j in range(p["J"]):
        nxt = pol[j, idx]
        cons.append(R * a[idx] + y[j] - a[nxt])
        assets.append(a[nxt]); idx = nxt
    return np.array(assets), np.array(cons)

assets, cons = simulate_profile(sol, p)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ages, sol["y"], "--", color="grey", label="income"); axes[0].plot(ages, cons, label="consumption"); axes[0].legend()
axes[1].plot(ages, assets[:-1]); axes[1].set_ylabel("assets")
for ax in axes: ax.set_xlabel("age")
plt.show()
print(f"beta * R = {p['beta'] * p['R']:.4f}; peak assets {assets.max():.2f} at age {ages[np.argmax(assets[:-1])]}")

## Task 5: a pension contribution

Set $\tau = 0.1$ and re-solve. What happens to consumption while working, to peak assets, and to consumption in retirement (the pension here does not change with $\tau$; in the project it will)?

In [ ]:
# SOLUTION
p1 = {**p, "tau": 0.1}
sol1 = solve_household_grid(p1)
assets1, cons1 = simulate_profile(sol1, p1)
print(f"working consumption: {cons[:45].mean():.3f} -> {cons1[:45].mean():.3f}")
print(f"peak assets:         {assets.max():.2f} -> {assets1.max():.2f}")
print(f"retired consumption: {cons[45:].mean():.3f} -> {cons1[45:].mean():.3f}")

## Task 6: the check

With $J = 2$, log utility, incomes $y_1 = 1$, $y_2 = 0.3$, $R = 1.04$, $\beta = 0.96$, the closed form is $c_1 = (y_1 + y_2/R)/(1+\beta)$ and $s = y_1 - c_1$. Solve the two-period problem on grids of 11, 101, 1001 points and compare the saving you get to the closed form. What is the error proportional to?

In [ ]:
# SOLUTION
y1, y2, R, beta = 1.0, 0.3, 1.04, 0.96
c1 = (y1 + y2 / R) / (1 + beta); s_cf = y1 - c1
for n in (11, 101, 1001):
    ag = np.linspace(0, 1, n)
    val = np.full(n, -np.inf); ok = y1 - ag > 0
    val[ok] = np.log(y1 - ag[ok]) + beta * np.log(R * ag[ok] + y2)
    print(f"n = {n:5d}: grid s = {ag[val.argmax()]:.4f}, closed form {s_cf:.4f}, error {abs(ag[val.argmax()] - s_cf):.4f}, spacing {1 / (n - 1):.4f}")

## Before next week

- This notebook is the skeleton of **M1 Part A**. The solver you tag (`m1-solo`, Friday 23 October) must add something the lecture did not build; the specification on Blackboard says what.
- **M0 is due Friday 2 October, 17:00.** A clean clone that builds, the Solow module with its tests, the golden rule, `AI_LOG.md`, and your commits.
- Read method card 3 (backward induction).
- Workflow note: if you have not yet, step through your Task 2 solver in the VS Code debugger (the note under Task 2). *Print statements are a debugger you have to clean up afterwards.*

# Tutorial 1

## Exercise 1: the grid

Re-solve with `n_a` = 30, 100, 300 and 1000. Time each solve (`%time`), record peak assets and the age of the peak, and plot assets by age for all four on one figure.

In [ ]:
# Exercise 1


## Exercise 2: patience and smoothing

Re-solve with $eta \in \{0.90, 0.96, 0.99\}$ and $\sigma \in \{1, 5\}$. Plot consumption by age; measure the slope of consumption between ages 41 and 60 and compare it with $(eta R)^{1/\sigma} - 1$.

In [ ]:
# Exercise 2


## Exercise 3

On paper, from the handout.